# Mapeamento do Banco de Dados Industrial

## Objetivo
Investigar a estrutura do banco de dados legado fornecido
para o PBL, identificando tabelas, colunas, tipos de dados,
chaves primárias, chaves estrangeiras e relacionamentos.

## Ambiente
- SGBD: PostgreSQL 18
- Banco: pbl_industria_homologacao
- Schema: industria
- Ferramentas: Python, Jupyter, psycopg e pandas

In [1]:
from pathlib import Path
import sys
import pandas as pd

raiz = Path.cwd().resolve()

if raiz.name == "notebooks":
    raiz = raiz.parent

sys.path.insert(0, str(raiz / "aplicacao"))

from database import conectar

print("Ambiente preparado!")

Ambiente preparado!


In [2]:
sql = """
SELECT
    table_name AS tabela,
    ordinal_position AS posicao,
    column_name AS coluna,
    data_type AS tipo_dado,
    is_nullable AS permite_nulo
FROM information_schema.columns
WHERE table_schema = 'industria'
ORDER BY table_name, ordinal_position;
"""

with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)

        registros = cur.fetchall()
        colunas = [desc.name for desc in cur.description]

estrutura = pd.DataFrame(registros, columns=colunas)

display(estrutura)

print(f"Tabelas identificadas: {estrutura['tabela'].nunique()}")
print(f"Total de colunas: {len(estrutura)}")

,tabela,posicao,coluna,tipo_dado,permite_nulo
0,auditoria,1,id,bigint,NO
1,auditoria,2,entidade,character varying,NO
2,auditoria,3,registro_id,bigint,NO
3,auditoria,4,operacao,character varying,NO
4,auditoria,5,dados_anteriores,jsonb,YES
...,...,...,...,...,...
69,servicos_ordem,5,executado_em,timestamp with time zone,NO
70,tecnicos,1,id,bigint,NO
71,tecnicos,2,matricula,character varying,NO
72,tecnicos,3,nome,character varying,NO


Tabelas identificadas: 12
Total de colunas: 74


Consultar as chaves primárias

In [3]:
sql = """
SELECT
    tc.table_name AS tabela,
    tc.constraint_name AS chave_primaria,
    kcu.column_name AS coluna,
    kcu.ordinal_position AS posicao
FROM information_schema.table_constraints AS tc
JOIN information_schema.key_column_usage AS kcu
    ON tc.constraint_catalog = kcu.constraint_catalog
    AND tc.constraint_schema = kcu.constraint_schema
    AND tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
    AND tc.table_name = kcu.table_name
WHERE tc.table_schema = 'industria'
    AND tc.constraint_type = 'PRIMARY KEY'
ORDER BY tc.table_name, kcu.ordinal_position;
"""

with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
        registros = cur.fetchall()
        nomes_colunas = [desc.name for desc in cur.description]

chaves_primarias = pd.DataFrame(
    registros,
    columns=nomes_colunas
)

display(chaves_primarias)

print(
    "Tabelas com PK:",
    chaves_primarias["tabela"].nunique()
)

,tabela,chave_primaria,coluna,posicao
0,auditoria,auditoria_pkey,id,1
1,componentes,componentes_pkey,id,1
2,depositos,depositos_pkey,id,1
3,equipamentos,equipamentos_pkey,id,1
4,estoques,estoques_pkey,deposito_id,1
5,estoques,estoques_pkey,componente_id,2
6,fornecedores,fornecedores_pkey,id,1
7,importacoes,importacoes_pkey,id,1
8,itens_ordem,itens_ordem_pkey,id,1
9,movimentacoes,movimentacoes_pkey,id,1


Tabelas com PK: 12


Identificar as chaves estrangeiras (FK)

In [4]:
sql = """
SELECT
    origem.relname AS tabela_origem,
    coluna_origem.attname AS coluna_fk,
    destino.relname AS tabela_referenciada,
    coluna_destino.attname AS coluna_pk,
    c.conname AS restricao
FROM pg_constraint c

JOIN pg_class origem
    ON origem.oid = c.conrelid

JOIN pg_namespace ns
    ON ns.oid = origem.relnamespace

JOIN pg_class destino
    ON destino.oid = c.confrelid

JOIN LATERAL unnest(c.conkey, c.confkey)
    WITH ORDINALITY AS pares(
        atributo_origem,
        atributo_destino,
        posicao
    ) ON TRUE

JOIN pg_attribute coluna_origem
    ON coluna_origem.attrelid = origem.oid
    AND coluna_origem.attnum = pares.atributo_origem

JOIN pg_attribute coluna_destino
    ON coluna_destino.attrelid = destino.oid
    AND coluna_destino.attnum = pares.atributo_destino

WHERE c.contype = 'f'
    AND ns.nspname = 'industria'

ORDER BY
    origem.relname,
    c.conname,
    pares.posicao;
"""

with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
        registros = cur.fetchall()
        nomes = [desc.name for desc in cur.description]

chaves_estrangeiras = pd.DataFrame(
    registros,
    columns=nomes
)

display(chaves_estrangeiras)

print(
    "Total de restrições FK:",
    chaves_estrangeiras["restricao"].nunique()
)

,tabela_origem,coluna_fk,tabela_referenciada,coluna_pk,restricao
0,componentes,fornecedor_id,fornecedores,id,componentes_fornecedor_id_fkey
1,estoques,componente_id,componentes,id,estoques_componente_id_fkey
2,estoques,deposito_id,depositos,id,estoques_deposito_id_fkey
3,itens_ordem,componente_id,componentes,id,itens_ordem_componente_id_fkey
4,itens_ordem,ordem_servico_id,ordens_servico,id,itens_ordem_ordem_servico_id_fkey
5,movimentacoes,componente_id,componentes,id,movimentacoes_componente_id_fkey
6,movimentacoes,deposito_id,depositos,id,movimentacoes_deposito_id_fkey
7,movimentacoes,importacao_id,importacoes,id,movimentacoes_importacao_id_fkey
8,movimentacoes,ordem_servico_id,ordens_servico,id,movimentacoes_ordem_servico_id_fkey
9,ordens_servico,equipamento_id,equipamentos,id,ordens_servico_equipamento_id_fkey


Total de restrições FK: 12


Consultar CHECK e UNIQUE

In [5]:
sql = """
SELECT
    t.relname AS tabela,
    c.conname AS restricao,
    CASE c.contype
        WHEN 'c' THEN 'CHECK'
        WHEN 'u' THEN 'UNIQUE'
    END AS tipo,
    pg_get_constraintdef(c.oid) AS definicao
FROM pg_constraint c
JOIN pg_class t
    ON t.oid = c.conrelid
JOIN pg_namespace n
    ON n.oid = t.relnamespace
WHERE n.nspname = 'industria'
    AND c.contype IN ('c', 'u')
ORDER BY t.relname, c.contype, c.conname;
"""

with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
        registros = cur.fetchall()
        colunas = [desc.name for desc in cur.description]

restricoes = pd.DataFrame(
    registros,
    columns=colunas
)

display(restricoes)

print(f"Total de restrições: {len(restricoes)}")

,tabela,restricao,tipo,definicao
0,componentes,componentes_codigo_key,UNIQUE,UNIQUE (codigo)
1,depositos,depositos_codigo_key,UNIQUE,UNIQUE (codigo)
2,equipamentos,equipamentos_tag_key,UNIQUE,UNIQUE (tag)
3,fornecedores,fornecedores_cnpj_key,UNIQUE,UNIQUE (cnpj)
4,ordens_servico,ordens_servico_numero_key,UNIQUE,UNIQUE (numero)
5,tecnicos,tecnicos_matricula_key,UNIQUE,UNIQUE (matricula)


Total de restrições: 6


### Análise das restrições CHECK e UNIQUE

A consulta aos metadados do PostgreSQL identificou seis restrições UNIQUE e nenhuma restrição CHECK declarada nas tabelas do schema industria.

As restrições UNIQUE estão presentes nas tabelas componentes, depositos, equipamentos, fornecedores, ordens_servico e tecnicos.

Na tabela movimentacoes, não foi identificada uma restrição CHECK sobre a quantidade movimentada, o que constitui uma hipótese relevante para a investigação do INC-006.

A tabela importacoes também não apresentou uma restrição UNIQUE, observação relevante para o INC-009.

Esses resultados ainda não comprovam os defeitos relatados, sendo necessário analisar as demais restrições, os índices, as validações da aplicação e os registros armazenados.

Investigar as restrições NOT NULL

In [6]:
sql = """
SELECT
    table_name AS tabela,
    column_name AS coluna,
    data_type AS tipo_dado,
    is_nullable AS permite_nulo
FROM information_schema.columns
WHERE table_schema = 'industria'
    AND is_nullable = 'NO'
ORDER BY table_name, ordinal_position;
"""

with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
        registros = cur.fetchall()
        nomes = [desc.name for desc in cur.description]

colunas_obrigatorias = pd.DataFrame(
    registros,
    columns=nomes
)

display(colunas_obrigatorias)

print(
    "Total de colunas que não aceitam NULL:",
    len(colunas_obrigatorias)
)

,tabela,coluna,tipo_dado,permite_nulo
0,auditoria,id,bigint,NO
1,auditoria,entidade,character varying,NO
2,auditoria,registro_id,bigint,NO
3,auditoria,operacao,character varying,NO
4,auditoria,executado_por,text,NO
...,...,...,...,...
56,servicos_ordem,executado_em,timestamp with time zone,NO
57,tecnicos,id,bigint,NO
58,tecnicos,matricula,character varying,NO
59,tecnicos,nome,character varying,NO


Total de colunas que não aceitam NULL: 61


### Análise das restrições NOT NULL

Foram identificadas 61 colunas com restrições NOT NULL
nas tabelas do schema industria.

Essas restrições impedem o armazenamento de valores
nulos, mas não garantem que valores numéricos estejam
dentro dos limites permitidos pelas regras de negócio.

A investigação será complementada pela análise das
colunas críticas, das demais restrições e das
validações existentes na aplicação Python.

Verificar as colunas críticas

In [7]:
sql = """
SELECT
    table_name AS tabela,
    column_name AS coluna,
    data_type AS tipo,
    is_nullable AS permite_nulo
FROM information_schema.columns
WHERE table_schema = 'industria'
  AND (
      (table_name = 'movimentacoes'
       AND column_name IN ('quantidade', 'tipo'))
      OR
      (table_name = 'estoques'
       AND column_name = 'quantidade')
      OR
      (table_name = 'importacoes'
       AND column_name = 'chave_lote')
  )
ORDER BY table_name, column_name;
"""

with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)

        registros = cur.fetchall()
        colunas = [desc.name for desc in cur.description]

colunas_criticas = pd.DataFrame(
    registros,
    columns=colunas
)

display(colunas_criticas)

,tabela,coluna,tipo,permite_nulo
0,estoques,quantidade,integer,NO
1,importacoes,chave_lote,character varying,NO
2,movimentacoes,quantidade,integer,NO
3,movimentacoes,tipo,character varying,NO


Consultar os índices existentes

In [8]:
sql = """
SELECT
    tablename AS tabela,
    indexname AS nome_indice,
    indexdef AS definicao
FROM pg_indexes
WHERE schemaname = 'industria'
ORDER BY tablename, indexname;
"""

with conectar() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)

        registros = cur.fetchall()
        colunas = [desc.name for desc in cur.description]

indices = pd.DataFrame(
    registros,
    columns=colunas
)

pd.set_option("display.max_colwidth", None)

display(indices)

print(f"Total de índices: {len(indices)}")

,tabela,nome_indice,definicao
0,auditoria,auditoria_pkey,CREATE UNIQUE INDEX auditoria_pkey ON industria.auditoria USING btree (id)
1,componentes,componentes_codigo_key,CREATE UNIQUE INDEX componentes_codigo_key ON industria.componentes USING btree (codigo)
2,componentes,componentes_pkey,CREATE UNIQUE INDEX componentes_pkey ON industria.componentes USING btree (id)
3,depositos,depositos_codigo_key,CREATE UNIQUE INDEX depositos_codigo_key ON industria.depositos USING btree (codigo)
4,depositos,depositos_pkey,CREATE UNIQUE INDEX depositos_pkey ON industria.depositos USING btree (id)
5,equipamentos,equipamentos_pkey,CREATE UNIQUE INDEX equipamentos_pkey ON industria.equipamentos USING btree (id)
6,equipamentos,equipamentos_tag_key,CREATE UNIQUE INDEX equipamentos_tag_key ON industria.equipamentos USING btree (tag)
7,estoques,estoques_pkey,"CREATE UNIQUE INDEX estoques_pkey ON industria.estoques USING btree (deposito_id, componente_id)"
8,fornecedores,fornecedores_cnpj_key,CREATE UNIQUE INDEX fornecedores_cnpj_key ON industria.fornecedores USING btree (cnpj)
9,fornecedores,fornecedores_pkey,CREATE UNIQUE INDEX fornecedores_pkey ON industria.fornecedores USING btree (id)


Total de índices: 22


### Análise dos índices

Foram identificados 22 índices no schema industria,
sendo 12 associados às chaves primárias, 6 às
restrições UNIQUE e 4 índices adicionais.

Na tabela movimentacoes, foram encontrados índices
sobre as colunas id e ordem_servico_id.

Não foram identificados índices específicos sobre
componente_id ou momento.

Essa observação constitui uma hipótese para a
investigação do INC-001. A confirmação dependerá
da análise dos planos de execução e da comparação
do desempenho em diferentes volumes de dados.